# Tour the `orinth` SDK

One cell per module. Nothing here trains or writes anything you have to clean
up — it is the map, and every other template is a route across it.

| Module | What it is |
| --- | --- |
| `orinth.datasets` | The catalog, and the rows inside it |
| `orinth.models` | The registry, and the predictors behind it |
| `orinth.train` | Start a real training job on the platform |
| `orinth.evaluate` | Score a model against a split, recorded |
| `orinth.inference` | Predict one input, recorded |
| `orinth.runs` | Log metrics into the rail beside this notebook |
| `orinth.projects` / `orinth.settings` | Where you are, and how it is configured |

In [ ]:
import orinth

print("orinth", orinth.__version__)
print("project", orinth.project().id, "|", orinth.project().task_types)
print("workspace", orinth.settings.workspace()["storage"])

## `orinth.datasets`

`list()` and `get()` hand back frozen `DatasetRef`s — `readiness` is the phase-21
verdict verbatim, and `bool(ref.readiness)` is `trainable`.

In [ ]:
for dataset in orinth.datasets.list()[:8]:
    ready = "ready" if dataset.readiness else "not trainable"
    print(f"{dataset.id:42} {dataset.task_type:20} {dataset.total_items:>6} items  {ready}")

In [ ]:
# Rows, as a polars frame. `records()` streams instead, for a split too large
# to hold; `images()` yields (name, PIL.Image); `paths()` gives files on disk.
ref = orinth.datasets.list()[0]
frame = orinth.datasets.load(ref.id, "train", limit=5)
print(ref.name, frame.shape)
frame.head(3)

## `orinth.models`

`predictor()` returns the object `/inference` runs — same weights, same
pre-processing — so anything measured with it is what the platform will do.

In [ ]:
for model in orinth.models.list()[:8]:
    print(f"{model.id:42} {model.family:22} {model.task_type:20} available={model.available}")

MODEL_ID = next(model.id for model in orinth.models.list() if model.available)
print("\nartifacts of", MODEL_ID)
print(orinth.models.artifacts(MODEL_ID))
print("inference defaults:", orinth.models.parameters())

## `orinth.train`

Starts **the platform's own job**. It appears on `/training`, registers a model
the catalog knows about, and outlives this notebook — which is exactly why it
goes over HTTP rather than fitting a model in this kernel.

Listing and inspecting are free; `start()` is the one call that does something,
so it is left commented out here.

In [ ]:
compute = orinth.train.compute()
for framework in compute["frameworks"]:
    devices = ", ".join(device["name"] for device in framework["devices"]) or "cpu only"
    print(f"{framework['name']:12} {str(framework['version']):10} accelerated={framework['accelerated']}  {devices}")

print()
for option in orinth.train.options("classification")[:5]:
    print(f"{option['id']:32} {option['family']:22} runnable={option['runnable']}")

print()
for run in orinth.train.list()[:5]:
    print(f"{run.id[:12]} {run.status:12} {run.percent:5.1f}%  {run.model_id or '-'}")

# run = orinth.train.start(dataset_id, model_option_id="keras_mobilenet_v2",
#                          task_type="classification", epochs=10)
# run = orinth.train.wait_for(run.id)

## `orinth.evaluate`

A recorded score, on `/testing`, comparable against every other evaluation of
the same split. `datasets()` lists the keys it takes — a key names a dataset
*and* a split, which is why it is not just a dataset id.

In [ ]:
for entry in orinth.evaluate.datasets()[:5]:
    print(f"{entry['key']:52} {entry['task_type']:20} available={entry['available']}")

for job in orinth.evaluate.list()[:5]:
    print(f"{job.id[:12]} {job.model_id:38} {job.status}")

# job = orinth.evaluate.start(MODEL_ID, "dataset:<id>:test", limit=50)
# print(job.metrics["image"]["overall"])

## `orinth.inference`

Two ways to predict, and they answer different questions:

- `orinth.models.predictor(id).predict(path, params)` runs in **this kernel**.
  Fast in a loop, records nothing — that is the batch template's tool.
- `orinth.inference.predict(...)` posts to the API. Slower per call, and the
  result is **recorded**: an id, an overlay, a row on `/inference`.

In [ ]:
image = orinth.datasets.paths(ref.id, "test")[0] if ref.splits.get("test") else None
print("would predict on:", image.name if image else "no test split on this dataset")

for prediction in orinth.inference.list(limit=5):
    print(f"{prediction.id[:12]} {prediction.model_id:38} -> {prediction.label}")

# prediction = orinth.inference.predict(MODEL_ID, image)
# print(prediction.label, prediction.scores)

## `orinth.runs`

Metrics from this notebook, in the rail on the right. `log()` starts a run on
first call, so the shortest useful thing is one line.

In [ ]:
with orinth.runs.start("tour", params={"note": "smoke"}) as run:
    for step in range(3):
        run.log(step, value=float(step) ** 0.5)
    run.log_text("what", "a three-point run, so the rail has something to draw")

print(run.id, "->", [entry.name for entry in orinth.runs.list()][:5])